# Stage 2: MotionGPT -- Text-to-Motion Transformer

**This notebook is prepared, not yet run for real training** -- it requires the main VQ-VAE (`02_train_bones.ipynb`, `checkpoints/vqvae_bones.pt`) trained on the current 63-dim bones-only, `num_codes=2048` representation. That run needs to be (re)done first since both the feature definition (translation moved out to the separate root-motion net) and the codebook size have changed since any earlier checkpoint. The pipeline below has been verified end-to-end with a standalone smoke test (untrained VQ-VAE, tiny data slice, 3 epochs) -- plumbing works, but no real training has happened yet.

**Architecture**: literal GPT-2-style decoder-only transformer (`src/model/motion_gpt.py`) -- pre-LN causal self-attention + GELU MLP blocks, learned positional embeddings, weight-tied output head. No architectural inventions.

**What's custom** (everything else stays standard GPT):
- **Single shared vocabulary** spanning text tokens + motion codebook tokens (`src/data/gpt_dataset.py`) -- one `nn.Embedding` table, not two separate ones:
  - `[0, vocab_text_size)` -- text tokens (word-level, built from training captions, includes `<pad>`=0, `<unk>`=1)
  - `vocab_text_size` -- `MOTION_START`, the condition token marking "stop conditioning on text, start generating motion"
  - `vocab_text_size + 1` -- `EOS`
  - `[vocab_text_size + 2, vocab_text_size + 2 + num_codes)` -- motion codebook tokens (offset past the text vocab)
- **Loss masking**: the model is never supervised to predict text -- only `MOTION_START`, motion tokens, and `EOS`. Implemented as `loss_mask = target_ids >= vocab_text_size`, which conveniently also excludes padding (`pad_id=0 < vocab_text_size`) for free.
- **Motion tokenization**: each full (un-chunked) motion sequence is run through the frozen, trained VQ-VAE's encoder + quantizer to get its token sequence -- consistent with how the root-motion net avoids chunk-boundary artifacts by working on full sequences, not `chunk_len` windows.

**Text vocabulary**: plain word-level tokenization (lowercase, strip punctuation) built from the training captions only (`data/vocab/text_vocab.json`), `min_freq=1` (3585 words) -- no subword/BPE, kept simple for a first pass. Each caption is a separate (text, motion) training example, so a motion with 5 captions contributes 5 examples sharing the same motion tokens.

**`num_codes=2048`** (matches `02_train_bones.ipynb`, reduced from `10000`): the codebook-size ablation found 10k gave the best raw VQ-VAE reconstruction quality, but its sparsity loss ceiling (`-ln(10000)`) took too long to approach in a real CPU training run. `2048` was chosen over an even smaller size because it's directly ablation-tested (2021/2048 active, 98.7%, perplexity 1558.6 at 200 seqs), not an untested guess. This also shrinks the combined GPT vocabulary from ~13,600 to ~5,635 (3585 text words + 2048 motion codes + 2 special tokens) against ~5,400 training sequences -- a more favorable ratio for the open question below, though that wasn't the reason for the change.

**Known open question (already discussed, not yet resolved)**: whether a larger motion codebook (which the earlier ablation showed reconstructs better) hurts the GPT's ability to learn well vs. a smaller one, given limited training data, is exactly what this notebook is meant to let you test empirically once both are trained. The switch to `2048` was driven by training-time practicality, not this question, but it does make for an easier starting point.

In [1]:
import sys
from pathlib import Path
from functools import partial

sys.path.insert(0, "..")

import torch
from torch.utils.data import DataLoader

from src.data.splits import load_splits
from src.data.normalize import NormStats
from src.data.text_vocab import Vocab
from src.data.gpt_dataset import (
    MotionTextDataset,
    collate_gpt,
    precompute_motion_tokens,
    save_motion_tokens,
    load_motion_tokens,
)
from src.model.vqvae import VQVAE
from src.model.motion_gpt import MotionGPT
from src.train_gpt import train_gpt


## Text vocabulary

Build (or load, if already built) the word-level vocabulary from training captions.

In [2]:
ROOT = Path.cwd().parent
raw_dir = ROOT / "data/raw/motion_text_7000"
splits = load_splits(ROOT / "data/splits")

vocab_path = ROOT / "data/vocab/text_vocab.json"
if vocab_path.exists():
    vocab = Vocab.load(vocab_path)
else:
    vocab = Vocab.build(splits["train"], raw_dir, min_freq=1)
    vocab_path.parent.mkdir(parents=True, exist_ok=True)
    vocab.save(vocab_path)

print("text vocab size:", len(vocab))


text vocab size: 3585


## Load the frozen, trained VQ-VAE

Motion tokenization uses this model's encoder + quantizer, no gradient.

In [3]:
device = torch.device("cpu")

FEATURE_DIM = 63  # bones-only, body-only (see 02_train_bones.ipynb)
NUM_CODES = 2048  # matches 02_train_bones.ipynb's current config (reduced from
                   # 10000 -- sparsity took too long to reach its ceiling on CPU;
                   # 2048 is directly ablation-tested, not just a guess)
CODE_DIM = 512

stats_bones = NormStats.load(ROOT / "data/stats/stats_bones_only_body.npz")

vqvae = VQVAE(
    feature_dim=FEATURE_DIM, code_dim=CODE_DIM, num_codes=NUM_CODES,
    commitment_beta=0.25, sparsity_weight=0.1, sparsity_temperature=0.5, w_vel=0.5,
).to(device)

vqvae_ckpt = ROOT / "checkpoints/vqvae_bones.pt"
if vqvae_ckpt.exists():
    vqvae.load_state_dict(torch.load(vqvae_ckpt, map_location=device))
    print("loaded trained VQ-VAE from", vqvae_ckpt)
else:
    print("WARNING: no VQ-VAE checkpoint found -- using an UNTRAINED model. "
          "Motion tokens will be meaningless until 02_train_bones.ipynb is (re)run "
          "with the current 63-dim config and this cell is re-run.")

vqvae.eval()
for p in vqvae.parameters():
    p.requires_grad_(False)


loaded trained VQ-VAE from /Users/bereket/Desktop/Jupyter/metrina-vqvae/checkpoints/vqvae_bones.pt


## Pre-tokenize all motion data (expensive -- run once, cached to disk)

Motion tokenization means running the VQ-VAE encoder, which is the expensive part of this whole pipeline -- so it happens **once, up front, for every sequence in train+val+test, before training starts**, not lazily per-example or per-epoch inside the training loop. Results are cached to `data/motion_tokens/motion_tokens_{num_codes}.json`; if that file already exists (e.g. a previous run of this cell), it's loaded directly and the VQ-VAE encoder isn't touched at all. The cache stores raw codebook indices (not yet offset into the combined GPT vocabulary), so it stays valid even if the text vocabulary changes.

Text tokenization (word-level, regex + dict lookup) is comparatively free, but is also fully precomputed -- along with the final `[text, MOTION_START, motion, EOS]` sequences themselves -- inside `MotionTextDataset.__init__` in the next cell, so `__getitem__` during training does zero computation, just a list lookup.

In [4]:
all_ids = splits["train"] + splits["val"] + splits["test"]

motion_tokens_path = ROOT / f"data/motion_tokens/motion_tokens_{NUM_CODES}.json"
if motion_tokens_path.exists():
    motion_tokens = load_motion_tokens(motion_tokens_path)
    print(f"loaded cached motion tokens for {len(motion_tokens)} sequences from {motion_tokens_path}")
else:
    print(f"tokenizing {len(all_ids)} sequences (running the VQ-VAE encoder once per sequence)...")
    motion_tokens = precompute_motion_tokens(all_ids, raw_dir, vqvae, stats_bones, device)
    save_motion_tokens(motion_tokens, motion_tokens_path)
    print(f"done, cached to {motion_tokens_path}")

token_lens = [len(v) for v in motion_tokens.values()]
print(f"motion tokens per sequence: mean={sum(token_lens)/len(token_lens):.1f}, "
      f"min={min(token_lens)}, max={max(token_lens)}")


tokenizing 6009 sequences (running the VQ-VAE encoder once per sequence)...
done, cached to /Users/bereket/Desktop/Jupyter/metrina-vqvae/data/motion_tokens/motion_tokens_2048.json
motion tokens per sequence: mean=32.3, min=6, max=50


## Text-motion dataset

In [5]:
train_ds = MotionTextDataset(splits["train"], raw_dir, vocab, motion_tokens, NUM_CODES)
val_ds = MotionTextDataset(splits["val"], raw_dir, vocab, motion_tokens, NUM_CODES)
test_ds = MotionTextDataset(splits["test"], raw_dir, vocab, motion_tokens, NUM_CODES)

layout = train_ds.layout
print("vocab layout:", layout)
print(f"train: {len(train_ds)} (text, motion) examples")
print(f"val:   {len(val_ds)} examples")
print(f"test:  {len(test_ds)} examples")


vocab layout: {'vocab_text_size': 3585, 'motion_start_id': 3585, 'eos_id': 3586, 'motion_token_offset': 3587, 'vocab_size': 5635}
train: 16170 (text, motion) examples
val:   897 examples
test:  901 examples


In [6]:
BATCH_SIZE = 16

collate = partial(collate_gpt, vocab_text_size=layout["vocab_text_size"])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

input_ids, target_ids, loss_mask = next(iter(train_loader))
print("input_ids:", input_ids.shape, "target_ids:", target_ids.shape, "loss_mask:", loss_mask.shape)
print("supervised (non-text, non-pad) fraction:", loss_mask.float().mean().item())


input_ids: torch.Size([16, 79]) target_ids: torch.Size([16, 79]) loss_mask: torch.Size([16, 79])
supervised (non-text, non-pad) fraction: 0.4280063211917877


## Model setup

In [7]:
MAX_SEQ_LEN = 256  # generous headroom -- mean motion tokens/sequence ~26, captions typically well under 100 words

model = MotionGPT(
    vocab_size=layout["vocab_size"],
    max_seq_len=MAX_SEQ_LEN,
    d_model=256,
    n_layers=6,
    n_heads=8,
    mlp_ratio=4,
    dropout=0.1,
    pad_id=0,
).to(device)

print("num params:", sum(p.numel() for p in model.parameters()))


num params: 6247168


## Training

**Not run yet** -- waiting on the main VQ-VAE to finish (re)training on the 63-dim config first.

In [9]:
EPOCHS = 1
LR = 3e-4

history = train_gpt(model, train_loader, val_loader, layout["vocab_text_size"], device, epochs=EPOCHS, lr=LR, log_every=1)

checkpoint_path = ROOT / "checkpoints/motion_gpt.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print("saved to", checkpoint_path)


epoch    0 | train_loss=0.6156 val_loss=2.7442
saved to /Users/bereket/Desktop/Jupyter/metrina-vqvae/checkpoints/motion_gpt.pt
